# Forward Pass Debugging: Layer-by-Layer Reference

This notebook performs a forward pass through a .keras model one layer at a time, for the purpose of verifying low-level implementations in C or RISC-V.

At each step, it prints the output of the current layer. These values act as ground truth for validating manual implementations. The output of one layer is passed directly as the input to the next, preserving the inference flow.

This setup helps identify discrepancies between the high-level model and its low-level counterparts, allowing for precise, layer-specific debugging.


In [1]:
import os
import tensorflow as tf
import numpy as np
import random

# Suppress TensorFlow warnings
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'


2025-05-14 04:50:40.967394: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-05-14 04:50:40.979947: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2025-05-14 04:50:41.095014: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2025-05-14 04:50:41.172169: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1747180241.240785   77214 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1747180241.26

## 1.0 Load and preprocess the data

In [2]:
(images, labels), _ = tf.keras.datasets.mnist.load_data()
images = images.astype("float32") / 255.0  # Normalize the images to [0, 1]
images = np.expand_dims(images, -1)  # Add channel dimension
labels = tf.keras.utils.to_categorical(labels, 10)  # One-hot encode the labels

## 2.0 Helper Functions

### 2.1 Function to Get a random image and label

Get a random image and label from the dataset. This function is used to generate a random input for the model.

In [3]:
def save_in_riscv_format(x, filename):
    with open(filename, 'w') as f:
        if len(x.shape) == 4:
            _, height, width, channels = x.shape
            for c in range(channels):
                for i in range(height):
                    row = [f"{x[0, i, j, c]:.6f}" for j in range(width)]
                    f.write(f".float " + ", ".join(row) + "\n")
                f.write("\n")
        
        elif len(x.shape) == 2:
            rows, cols = x.shape
            values = [f"{x[i, j]:.6f}" for i in range(rows) for j in range(cols)]
            f.write(f".float " + ", ".join(values) + "\n")
        
        else:
            raise ValueError("Unsupported shape")



def get_random_image(index=None, output_file="random_image.txt"):
    # Randomly select an image and its label
    if index is None:
        index = random.randint(0, len(images) - 1)
    image = images[index].squeeze()  # (28, 28)
    image = tf.expand_dims(image, axis=0)  # Add batch dimension (1, 28, 28)
    image = tf.expand_dims(image, axis=-1)  # Add channel dimension (1, 28, 28, 1)
    
    label = np.argmax(labels[index])  # Get label
    
    # Save the image to a .txt file with 3 decimal places
    with open(output_file, "w") as f:
        for row in image.numpy().squeeze():  # Convert tensor to numpy and remove extra dimensions
            row_str = ".float " + ", ".join(f"{val:.3f}" for val in row)  # Using commas to separate values
            f.write(row_str + "\n")
    
    return image, label

### 2.2 Function to Print a tensor

Prints the shape and values of a tensor in a readable format.

For 4D tensors (e.g., batches of images), it prints the values of the first sample,
channel by channel. For 2D tensors (e.g., dense layer outputs), it prints all values
row by row. Other shapes are not currently supported.

In [4]:
def print_shape_and_values(x):
    print(f"Shape: {x.shape}")
    
    if len(x.shape) == 4:
        _, height, width, channels = x.shape
        for c in range(channels):
            for i in range(height):
                row = [f"{x[0, i, j, c]:.3f}" for j in range(width)]
                print(", ".join(row))
            print()
    
    elif len(x.shape) == 2:
        rows, cols = x.shape
        for i in range(rows):
            row = [f"{x[i, j]:.3f}" for j in range(cols)]
            print(", ".join(row))
    else:
        print("Unsupported shape")


## 3.0 Load the model from mnist_cnn_model.keras

In [5]:
model = tf.keras.models.load_model("../models/mnist_cnn_model.keras")

2025-05-14 04:50:46.896506: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


## 4.0 Get a random image, label and step through the model layer by layer

### 4.1 Get a random image and label

In [12]:
image, label = get_random_image()
print(f"Label: {label}")

Label: 6


### 4.2 Step through the model layer by layer

#### 4.2.1 Input Image

In [13]:
print("Original Image:")
print_shape_and_values(image)

Original Image:
Shape: (1, 28, 28, 1)
0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000
0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000
0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000
0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.004, 0.471, 0.847, 1.000, 0.996, 0.412, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000
0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.008, 0.529, 0.996, 0.988, 0.780, 0.929, 0.286, 0.000, 0.000, 0.000, 0.000, 0.000, 

#### 4.2.2 Conv2D Layer

Note: Refer to section **4.2.1** for the input

In [14]:
conv2d_out = model.layers[0](image)
print_shape_and_values(conv2d_out)
save_in_riscv_format(conv2d_out, "conv2d_out.txt")

Shape: (1, 24, 24, 8)
-0.460, -0.460, -0.460, -0.460, -0.460, -0.460, -0.460, -0.460, -0.460, -0.460, -0.458, -0.329, -0.075, -0.003, -0.017, 0.098, 0.091, -0.054, -0.092, -0.276, -0.415, -0.460, -0.460, -0.460
-0.460, -0.460, -0.460, -0.460, -0.460, -0.460, -0.460, -0.460, -0.460, -0.458, -0.329, -0.072, -0.055, -0.297, -0.482, -0.323, -0.142, 0.061, 0.103, -0.134, -0.392, -0.460, -0.460, -0.460
-0.460, -0.460, -0.460, -0.460, -0.460, -0.460, -0.460, -0.460, -0.460, -0.391, -0.122, -0.049, -0.457, -0.752, -0.616, -0.411, -0.379, -0.339, -0.386, -0.541, -0.529, -0.460, -0.460, -0.460
-0.460, -0.460, -0.460, -0.460, -0.460, -0.460, -0.460, -0.460, -0.450, -0.253, -0.076, -0.507, -0.825, -0.476, -0.014, -0.066, -0.328, -0.544, -0.606, -0.777, -0.605, -0.460, -0.460, -0.460
-0.460, -0.460, -0.460, -0.460, -0.460, -0.460, -0.460, -0.460, -0.371, -0.130, -0.341, -0.820, -0.596, 0.035, 0.123, -0.242, -0.570, -0.641, -0.501, -0.616, -0.521, -0.460, -0.460, -0.460
-0.460, -0.460, -0.460, -0.46

#### 4.2.3 ReLU Activation

Note: Refer to section **4.2.2** for the input

In [15]:
relu_out = model.layers[1](conv2d_out)
print_shape_and_values(relu_out)
save_in_riscv_format(relu_out, "relu_out.txt")

Shape: (1, 24, 24, 8)
0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.098, 0.091, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000
0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.061, 0.103, 0.000, 0.000, 0.000, 0.000, 0.000
0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000
0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000
0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.035, 0.123, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000
0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.235, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.0

#### 4.2.4 MaxPooling

Note: Refer to section **4.2.3** for the input

In [16]:
maxpool_out = model.layers[2](relu_out)
print_shape_and_values(maxpool_out)
save_in_riscv_format(maxpool_out, "maxpool_out.txt")

Shape: (1, 12, 12, 8)
0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.098, 0.091, 0.103, 0.000, 0.000
0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000
0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.235, 0.123, 0.000, 0.000, 0.000, 0.000
0.000, 0.000, 0.000, 0.000, 0.000, 0.112, 0.248, 0.000, 0.000, 0.000, 0.000, 0.000
0.000, 0.000, 0.000, 0.000, 0.000, 0.576, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000
0.000, 0.000, 0.000, 0.000, 0.516, 0.548, 0.000, 0.475, 0.251, 0.000, 0.000, 0.000
0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000
0.000, 0.000, 0.000, 0.000, 0.294, 0.000, 0.000, 0.000, 0.000, 0.135, 0.000, 0.000
0.000, 0.000, 0.000, 0.000, 0.498, 0.047, 0.000, 0.000, 0.207, 0.187, 0.000, 0.000
0.000, 0.000, 0.000, 0.000, 0.000, 0.162, 0.000, 0.000, 0.093, 0.000, 0.000, 0.000
0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000
0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.

#### 4.2.5 Flatten

Note: Refer to section **4.2.4** for the input

In [17]:
flatten_out = model.layers[3](maxpool_out)
print_shape_and_values(flatten_out)
save_in_riscv_format(flatten_out, "flatten_out.txt")

Shape: (1, 1152)
0.000, 0.178, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.178, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.178, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.178, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.180, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.838, 0.000, 0.000, 0.000, 0.333, 0.093, 0.000, 0.000, 1.637, 0.000, 0.000, 0.000, 0.715, 0.512, 0.000, 0.098, 2.791, 0.760, 0.000, 0.425, 0.122, 0.861, 0.000, 0.091, 3.012, 1.395, 0.852, 0.204, 0.000, 0.859, 0.000, 0.103, 1.837, 0.822, 1.351, 0.118, 0.000, 0.367, 0.000, 0.000, 0.460, 0.000, 0.263, 0.000, 0.000, 0.000, 0.000, 0.000, 0.178, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.178, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.178, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.178, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.178, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.511, 0.000, 0.000, 0.000, 0.120, 0.000, 0.000, 0.000, 1.170, 0.000, 0.000, 0.0

#### 4.2.6 Fifth Layer: Dense Layer

Note: Refer to section **4.2.5** for the input

In [18]:
dense_out = model.layers[4](flatten_out)  # Fifth layer output
print_shape_and_values(dense_out)
save_in_riscv_format(dense_out, "dense_out.txt")

Shape: (1, 10)
-5.791, -7.480, -11.161, -9.319, -9.348, 3.909, 8.004, -22.609, -3.778, -10.471


#### 4.2.7 Layer Six: Softmax

Note: Refer to section **4.2.6** for the input

In [19]:
softmax_out = model.layers[5](dense_out)
print_shape_and_values(softmax_out)
save_in_riscv_format(softmax_out, "softmax_out.txt")

Shape: (1, 10)
0.000, 0.000, 0.000, 0.000, 0.000, 0.016, 0.984, 0.000, 0.000, 0.000


### 4.3 Get model prediction

In [20]:
print(f"Predicted class: {np.argmax(softmax_out)}")
print(f"True class: {label}")

Predicted class: 6
True class: 6
